In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Generate a manifest file (images.txt) with all blob names (or URLs) in an Azure container.
One line per blob.
"""
!pip install requests azure-storage-blob tqdm
import os
from azure.storage.blob import BlobServiceClient
from tqdm import tqdm
import urllib.parse

# -----------------------
# HARD-CODED CONFIG
# -----------------------
CONNECTION_STRING = ""

CONTAINER_NAME = "london"
PREFIX = "training/training/Images"
OUTPUT_FILE = "images.txt"

WRITE_URLS = False
URL_BASE = f"https://<youraccount>.blob.core.windows.net/{CONTAINER_NAME}"


def main():
    svc = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    container_client = svc.get_container_client(CONTAINER_NAME)

    total = 0
    with open(OUTPUT_FILE, "w", encoding="utf-8", newline="\n") as fh:
        for blob in tqdm(container_client.list_blobs(name_starts_with=PREFIX), desc="Listing blobs"):
            name = blob.name
            if WRITE_URLS:
                encoded = '/'.join(urllib.parse.quote(part, safe='') for part in name.split('/'))
                url = f"{URL_BASE.rstrip('/')}/{encoded}"
                fh.write(url + "\n")
            else:
                fh.write(name + "\n")
            total += 1

    print(f"Done. Wrote {total} blob entries to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()
